In [7]:
pip install transformers torch

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_attentions=True)
model.eval()

text = "Deep learning models understand language"


Loading weights: 100%|███████████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 3271.55it/s]


In [ ]:
# --------------------------------------------------
# Step 1: Tokenize the Input
# --------------------------------------------------
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['Deep', 'Ġlearning', 'Ġmodels', 'Ġunderstand']


In [16]:
# --------------------------------------------------
# Step 2: Inspect Token Embeddings
# --------------------------------------------------
token_embeddings = model.transformer.wte(inputs["input_ids"])
print("\nToken embedding shape:", token_embeddings.shape)
print("Embedding for first token (first 8 values):")
print(token_embeddings[0][0][:8])


Token embedding shape: torch.Size([1, 4, 768])
Embedding for first token (first 8 values):
tensor([-0.0385, -0.0472,  0.1723, -0.0468,  0.0374, -0.0831, -0.2232, -0.1254],
       grad_fn=<SliceBackward0>)


In [17]:
# --------------------------------------------------
# Step 3: Inspect Positional Encoding
# --------------------------------------------------
position_ids = torch.arange(inputs["input_ids"].shape[1]).unsqueeze(0)
position_embeddings = model.transformer.wpe(position_ids)
print("\nPositional embedding shape:", position_embeddings.shape)
print("Positional embedding for position 0 (first 8 values):")
print(position_embeddings[0][0][:8])


Positional embedding shape: torch.Size([1, 4, 768])
Positional embedding for position 0 (first 8 values):
tensor([-0.0188, -0.1974,  0.0040,  0.0113,  0.0638, -0.1050,  0.0369, -0.1680],
       grad_fn=<SliceBackward0>)


In [18]:
# --------------------------------------------------
# Step 4: Run the Model and Inspect Attention Weights
# --------------------------------------------------
with torch.no_grad():
    outputs = model(**inputs)

attentions = outputs.attentions  # one tensor per transformer layer
print("\nNumber of layers:", len(attentions))
print("Attention shape per layer (batch, heads, seq_len, seq_len):", attentions[0].shape)

layer1_head1 = attentions[0][0][0]
print("\nLayer 1, Head 1 attention weights (each row = one token's focus):")
print(layer1_head1)



Number of layers: 6
Attention shape per layer (batch, heads, seq_len, seq_len): torch.Size([1, 12, 4, 4])

Layer 1, Head 1 attention weights (each row = one token's focus):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.8188, 0.1812, 0.0000, 0.0000],
        [0.6468, 0.2121, 0.1410, 0.0000],
        [0.3806, 0.4094, 0.1470, 0.0630]])


In [13]:
# --------------------------------------------------
# Step 5: Next-Token Prediction
# --------------------------------------------------
logits = outputs.logits
next_token_logits = logits[0, -1]
top5 = torch.topk(next_token_logits, 5)

print("\nTop 5 next-token predictions:")
for score, idx in zip(top5.values, top5.indices):
    print(f"{tokenizer.decode([idx]):>15}  (score: {score:.2f})")



Top 5 next-token predictions:
            and  (score: -73.66)
              ,  (score: -74.24)
       learning  (score: -74.38)
              .  (score: -74.82)
     processing  (score: -75.42)
